# HySwash: Surrogate Modeling and Long-Term Reconstruction

In the previous notebook, we used SWASH to simulate a representative set of
offshore wave conditions selected with MDA.

In this notebook, we will use those simulations to train **surrogate models**
that reproduce the nearshore wave transformation at a fraction of the
computational cost.

We will then apply these surrogate models to a long time series of offshore
forcing conditions to reconstruct the corresponding **nearshore wave climate**.

In [1]:
import os
import os.path as op

import numpy as np
import pandas as pd
import plotly.express as px
import xarray as xr

## Load the SWASH simulations

We will start by loading the HySwash configuration, the MDA model, and the
postprocessed SWASH simulations generated in the previous notebook.

These simulations will provide the **training data** for the surrogate models.

In [2]:
root_dir = os.getcwd()
output_dir = op.join(root_dir, "output")
templates_dir = op.join(root_dir, "templates")
export_dir = op.join(root_dir, "export")

In [3]:
from bluemath_tk.core.io import load_model

mda = load_model(op.join(export_dir, "mda_model.pkl"))
swash_model = load_model(op.join(export_dir, "swash_model.pkl"))

depth_file = op.join(templates_dir, "depth.bot")
depth_array = np.loadtxt(depth_file)

postprocessed_output = xr.open_dataset(
    op.join(output_dir, "output_postprocessed.nc")
)

postprocessed_output

<xarray.Dataset> Size: 139kB
Dimensions:   (case_num: 7, Tsec: 450, Xp: 321)
Coordinates:
  * case_num  (case_num) int64 56B 0 1 3 4 5 6 9
  * Tsec      (Tsec) int64 4kB 0 1 2 3 4 5 6 7 ... 443 444 445 446 447 448 449
  * Xp        (Xp) int64 3kB 0 1 2 3 4 5 6 7 ... 313 314 315 316 317 318 319 320
    Yp        int64 8B ...
Data variables:
    Ru2       (case_num) float64 56B ...
    Runlev    (case_num, Tsec) float64 25kB ...
    Msetup    (case_num, Xp) float64 18kB ...
    Hrms      (case_num, Xp) float64 18kB ...
    Hs        (case_num, Xp) float64 18kB ...
    Hss       (case_num, Xp) float64 18kB ...
    ig        (case_num, Xp) float64 18kB ...
    Hvlf      (case_num, Xp) float64 18kB ...

## Build the surrogate model

The SWASH simulations provide spatially distributed results along the
cross-shore profile for a limited number of representative offshore conditions.

Our goal is to predict these spatial fields for **new offshore conditions**
without running SWASH again.

HySwash combines two techniques for this purpose:

1. **Principal Component Analysis (PCA)** reduces each spatial SWASH output to a
   small number of principal components.
2. **Radial Basis Functions (RBF)** interpolate the principal components as a
   function of the offshore forcing conditions.

For a new combination of offshore conditions, the RBF model predicts the
principal components, and the PCA transformation is inverted to reconstruct
the complete cross-shore field.

**SWASH simulations → PCA → RBF → predicted PCs → spatial reconstruction**

### Principal Component Analysis (PCA)

The postprocessed SWASH outputs contain one value at each cross-shore location.
Instead of building an independent surrogate model for every location, we use
PCA to represent these spatial fields with a smaller number of variables.

PCA identifies the dominant spatial patterns in the SWASH simulations and
represents each simulation through its corresponding **principal component
scores (PCs)**.

By retaining only the components that explain most of the variance, we reduce
the dimensionality of the problem while preserving the dominant spatial
variability.

In [4]:
from bluemath_tk.datamining.pca import PCA

pca = PCA()

pca.fit_transform(
    data=postprocessed_output,
    vars_to_stack=["Hs"],
    coords_to_stack=["Xp"],
    pca_dim_for_rows="case_num",
    value_to_replace_nans={"Hs": 0.0},
)

pca.save_model(
    model_path=op.join(export_dir, "pca_model.pkl"),
)

2026-09-18 21:52:56,068 - PCA - WARNING - Using 291 out of 321 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
2026-09-18 21:52:56,107 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!


### Spatial surrogate: RBF interpolation

Once the spatial fields have been represented by their principal components,
we need to relate those components to the offshore forcing conditions.

For this purpose, HySwash uses **Radial Basis Function (RBF) interpolation**.
The RBF model learns the relationship between:

**offshore forcing conditions → principal component scores**

For new offshore conditions, the RBF predicts the PCs, which can then be
transformed back through the PCA model to reconstruct the complete spatial field.

**offshore conditions → RBF → PCs → inverse PCA → spatial field**

In [5]:
from bluemath_tk.interpolation.rbf import RBF

rbf_spatial = RBF()

rbf_spatial.fit(
    subset_data=mda.centroids.iloc[postprocessed_output["case_num"].values],
    target_data=pca.pcs_df,
)

rbf_spatial.save_model(
    model_path=op.join(export_dir, "rbf_spatial_model.pkl")
)

### Scalar surrogate: wave runup

Some SWASH outputs, such as `Ru2`, contain a **single value for each
simulation**. In this case, dimensionality reduction is not required.

The RBF can therefore be trained directly between the offshore forcing
conditions and the scalar output:

**offshore conditions → RBF → `Ru2`**

This provides a simpler surrogate-modeling strategy for scalar quantities.

In [6]:
Ru2 = pd.DataFrame(
    postprocessed_output["Ru2"].values,
    columns=["Ru2"],
)

rbf_scalar = RBF()

rbf_scalar.fit(
    subset_data=mda.centroids.iloc[postprocessed_output["case_num"].values],
    target_data=Ru2,
)

rbf_scalar.save_model(
    model_path=op.join(export_dir, "rbf_scalar_model.pkl")
)

ValueError: Data contains NaNs.

## Reconstruction

Now that both surrogate models have been trained, we can apply them to new
offshore forcing conditions without running additional SWASH simulations.

We will use the **10,000 conditions generated with LHS** to illustrate the two
reconstruction strategies:

- **Scalar reconstruction:** predict `Ru2` directly with the scalar RBF.
- **Spatial reconstruction:** predict the PCs with the spatial RBF and use the
  inverse PCA transformation to recover the complete cross-shore field.

In [ ]:
lhs_dataset = pd.read_pickle(
    op.join(export_dir, "lhs_dataset.pkl")
)

### 1. Scalar reconstruction

For scalar outputs, the reconstruction is performed directly by the RBF model.

Here, we predict `Ru2` for all LHS forcing combinations.

In [ ]:
Ru2_reconstructed = rbf_scalar.predict(lhs_dataset)

lhs_reconstructed = lhs_dataset.copy()
lhs_reconstructed["Ru2"] = Ru2_reconstructed

In [ ]:
fig = px.scatter_3d(
    lhs_reconstructed,
    x="Hs",
    y="Hs_L0",
    z="WL",
    color="Ru2",
)

fig.update_traces(marker=dict(size=3))
fig.show()

### 2. Spatial reconstruction

Spatial outputs require an additional step.

First, the RBF model predicts the principal component scores for each new
offshore condition. The PCA transformation is then inverted to recover the
complete spatial field along the cross-shore profile.

**new forcing → predicted PCs → inverse PCA → reconstructed field**

In [ ]:
pcs_reconstructed = rbf_spatial.predict(lhs_dataset)

In [ ]:
pcs_reconstructed

,PC1,PC2
0,5.375894,49.618025
1,-9.244316,-133.223576
2,-14.110869,-49.466804
3,-4.335196,-48.812407
4,7.929291,-60.958513
...,...,...
8722,-8.558602,77.191646
8723,8.838680,-48.856915
8724,-4.693048,105.378980
8725,-19.449041,53.911567
